# Lab 07: Agent Framework Deep Dive

## 🎯 What We're Building

In Lab 01 you built a raw ReAct agent with just the OpenAI SDK. It worked — but you wrote 80+ lines of loop logic, tool dispatch, and message management yourself.

In this lab, you'll build **the exact same agent** using three production frameworks, and see how each one solves the same problems differently:

| Stage | Framework | What You Build | What You Learn |
|-------|-----------|----------------|----------------|
| **Stage 1** | **LangGraph** | ReAct agent with graph nodes + checkpointing | Graph-based orchestration, state persistence |
| **Stage 2** | **Microsoft Agent Framework** | Same agent with Kernel + Plugins | Enterprise SDK, Azure-native patterns |
| **Stage 3** | **Deep Agents** | Same agent with planning + filesystem tools | Autonomous context management, subagent spawning |
| **Stage 4** | **MCP** | Connect a universal tool server | Open standard for tool interoperability |
| **Stage 5** | **Comparison** | Side-by-side analysis of all three | Decision criteria for choosing a framework |

> **Prerequisites:** Complete [Lab 01](../lab-01-react-agent/README.md). Read [Chapter 16: Agent Frameworks](../../education/en/16-agent-frameworks.md).

---

## 🔧 Setup

First, we install all three frameworks and connect to Azure OpenAI.

**Why these specific packages?**
- `langgraph` + `langchain-openai` — LangGraph needs LangChain's Azure OpenAI connector to talk to our deployed model
- `semantic-kernel` — Microsoft's SDK (Semantic Kernel is the Python package name for what evolved into MAF)
- `deepagents` — LangChain's autonomous agent harness, built on top of LangGraph
- `mcp` — The Model Context Protocol SDK for building universal tool servers
- `python-dotenv` — Loads our `.env` file with Azure connection strings

In [ ]:
# ──────────────────────────────────────────────────────
# Install all three frameworks + MCP
#
# This installs:
# • langgraph + langchain-openai  → Stage 1 (LangGraph)
# • semantic-kernel               → Stage 2 (Microsoft Agent Framework)
# • deepagents                    → Stage 3 (Deep Agents)
# • mcp                          → Stage 4 (MCP tool server)
# • python-dotenv                → Load Azure credentials
# ──────────────────────────────────────────────────────
%pip install langgraph langchain-openai semantic-kernel deepagents mcp python-dotenv --quiet

### Connect to Azure OpenAI

All three frameworks will use the **same Azure OpenAI deployment** (GPT-4.1).
This is important — by using the same model, any differences we see in behavior
are caused by the **framework**, not the model.

We load credentials from the `.env` file created in Lab 00.

In [ ]:
import os
import json
import time
from dotenv import load_dotenv

# ──────────────────────────────────────────────────────
# Load Azure OpenAI credentials.
# All three frameworks will share these same credentials
# so we get a fair comparison.
# ──────────────────────────────────────────────────────
load_dotenv("../.env")

AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41")

print(f"✅ Azure OpenAI endpoint: {AZURE_ENDPOINT}")
print(f"   Deployment: {MODEL}")
print(f"   API version: {AZURE_API_VERSION}")

### Define Shared Tools

We define the **same three tools** from Lab 01. Every framework will use these
exact functions so we can compare apples to apples.

| Tool | What It Does | Why We Include It |
|------|-------------|-------------------|
| `get_weather` | Returns weather for a city | Simple single-input tool |
| `calculate` | Evaluates a math expression | Tests argument parsing |
| `search_knowledge` | Searches a mini knowledge base | Tests multi-word queries |

These are intentionally simple — the goal is to compare **frameworks**, not tools.

In [ ]:
# ──────────────────────────────────────────────────────
# Shared Tools — identical across all three frameworks.
#
# These are plain Python functions. Each framework will
# wrap them differently (LangGraph uses @tool decorator,
# SK uses @kernel_function, Deep Agents takes raw functions).
# ──────────────────────────────────────────────────────

def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    weather_data = {
        "tel aviv": {"temp": 28, "condition": "sunny", "humidity": 65},
        "new york": {"temp": 12, "condition": "cloudy", "humidity": 70},
        "london":   {"temp": 8,  "condition": "rainy",  "humidity": 85},
        "tokyo":    {"temp": 20, "condition": "clear",  "humidity": 55},
    }
    data = weather_data.get(city.lower(), {"temp": 15, "condition": "unknown", "humidity": 50})
    return f"Weather in {city}: {data['temp']}°C, {data['condition']}, humidity {data['humidity']}%"


def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Only supports basic arithmetic."""
    allowed_chars = set('0123456789+-*/.() ')
    if not all(c in allowed_chars for c in expression):
        return "Error: only mathematical expressions are allowed"
    try:
        result = eval(expression)
        return f"Result: {expression} = {result}"
    except Exception as e:
        return f"Error calculating: {e}"


KNOWLEDGE_BASE = [
    {"topic": "vacation policy", "content": "Employees get 22 vacation days per year. Unused days can carry over up to 5 days."},
    {"topic": "remote work", "content": "Employees can work remotely up to 3 days per week. Core hours 10am-4pm."},
    {"topic": "expense policy", "content": "Meals up to $50/day during travel. Flights must be economy. Hotel max $200/night."},
    {"topic": "onboarding", "content": "New employees have a 90-day onboarding. Week 1: HR orientation. Month 2-3: projects."},
]

def search_knowledge(query: str) -> str:
    """Search the company knowledge base for policies and procedures."""
    query_lower = query.lower()
    results = []
    for item in KNOWLEDGE_BASE:
        if any(word in query_lower for word in item["topic"].split()):
            results.append(f"[{item['topic']}]: {item['content']}")
    return "\n\n".join(results) if results else "No relevant information found."


# Quick smoke test
print("🌤️", get_weather("Tel Aviv"))
print("🔢", calculate("15 * 7 + 3"))
print("📚", search_knowledge("vacation"))
print("\n✅ All 3 shared tools working.")

### The Test Question

We'll ask every framework the **same multi-step question** to see how each one
handles tool selection, multi-tool chaining, and final answer synthesis.

This question deliberately requires **three different tools**:
1. `get_weather` → weather in Tokyo
2. `search_knowledge` → expense policy (daily meal limit)
3. `calculate` → 3 × $15 = $45 vs $50 limit

In [ ]:
# ──────────────────────────────────────────────────────
# The same question for ALL frameworks.
# This ensures a fair comparison.
# ──────────────────────────────────────────────────────
TEST_QUESTION = (
    "I'm planning a trip to Tokyo. What's the weather there? "
    "Also, what's the daily meal expense limit? "
    "If I eat 3 meals at $15 each, am I within budget?"
)

SYSTEM_PROMPT = (
    "You are a helpful assistant for Acme Corp employees. "
    "Use the available tools to answer questions about weather, "
    "company policies, and calculations."
)

print(f"📝 Test question: {TEST_QUESTION}")

---

# 🏗️ STAGE 1: LangGraph — The Graph-Based Agent

## What is LangGraph?

**LangGraph** models agents as **state graphs** — directed graphs where:
- **Nodes** are functions that read and write to a shared State
- **Edges** define the flow between nodes (can be conditional)
- **State** is a dictionary that flows through the entire graph

### Why LangGraph?

In Lab 01, you built the ReAct loop manually with a `for` loop and `if/else`. LangGraph replaces all of that with a **graph definition**:

```
Lab 01 (raw):                    LangGraph:
┌──────────────────┐            ┌──────────────────┐
│ for i in range:  │            │ graph.add_node() │ ← Define nodes
│   call LLM       │     →      │ graph.add_edge() │ ← Define flow
│   if tool_call:  │            │ agent.invoke()   │ ← Run it
│     execute tool │            └──────────────────┘
│   else: break    │            + Checkpointing FREE
└──────────────────┘            + Streaming FREE
  80+ lines                     + HITL FREE
                                  ~15 lines
```

### The `create_react_agent` shortcut

LangGraph provides a pre-built `create_react_agent()` that implements the entire
ReAct pattern (Think → Act → Observe → repeat) in a single function call.
Under the hood, it creates a StateGraph with two nodes ("agent" and "tools")
and conditional edges between them — exactly what you built manually in Lab 01.

We use `AzureChatOpenAI` as the LLM connector because our model is deployed
on Azure OpenAI (not the public OpenAI API).

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 1: LangGraph ReAct Agent
#
# What this cell does:
# 1. Creates an AzureChatOpenAI LLM connector
#    (this is LangChain's adapter for Azure-hosted models)
# 2. Wraps our plain Python functions with @tool decorator
#    (LangGraph needs type-annotated tool wrappers)
# 3. Calls create_react_agent() which builds the full
#    ReAct state graph: LLM node ↔ Tool node
#
# That's it — 15 lines replaces the entire 80-line
# raw agent from Lab 01.
# ──────────────────────────────────────────────────────

from langchain_openai import AzureChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

# Step 1: Connect to Azure OpenAI via LangChain's adapter
# This wraps our Azure deployment so LangGraph can use it
llm = AzureChatOpenAI(
    azure_deployment=MODEL,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
)

# Step 2: Wrap our shared functions as LangGraph tools.
# The @tool decorator reads the function's docstring and
# type hints to auto-generate the JSON schema that the
# LLM needs (the same schema we wrote manually in Lab 01).
@tool
def lg_get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return get_weather(city)

@tool
def lg_calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return calculate(expression)

@tool
def lg_search_knowledge(query: str) -> str:
    """Search the company knowledge base for policies and procedures."""
    return search_knowledge(query)

lg_tools = [lg_get_weather, lg_calculate, lg_search_knowledge]

# Step 3: Create the agent.
# create_react_agent() builds a complete StateGraph:
#   START → agent_node (LLM) → tool_node → agent_node → ... → END
# It handles the entire ReAct loop internally.
lg_agent = create_react_agent(
    model=llm,
    tools=lg_tools,
    prompt=SYSTEM_PROMPT,
)

print("✅ LangGraph agent created")
print(f"   Model: {MODEL}")
print(f"   Tools: {[t.name for t in lg_tools]}")

### Run the LangGraph Agent

Now we invoke the agent with our test question and measure:
- **Response quality** — did it use all three tools and combine the results?
- **Latency** — total wall-clock time
- **Token usage** — how many tokens were consumed?

The `invoke()` method runs the full graph from START to END.
The result contains all messages, including intermediate tool calls.

In [ ]:
# ──────────────────────────────────────────────────────
# Run LangGraph on the test question.
#
# What happens under the hood:
# 1. The graph starts at the "agent" node (LLM call)
# 2. LLM returns tool_calls → graph routes to "tools" node
# 3. Tools execute, results added to state
# 4. Graph routes back to "agent" node (LLM sees results)
# 5. Repeat until LLM returns text (no more tool_calls)
# 6. Graph reaches END → returns final state
# ──────────────────────────────────────────────────────

print("📊 STAGE 1: LangGraph Agent")
print("=" * 60)

start = time.time()

lg_result = lg_agent.invoke(
    {"messages": [{"role": "user", "content": TEST_QUESTION}]}
)

lg_time = time.time() - start

# Extract the final answer (last message in the conversation)
lg_answer = lg_result["messages"][-1].content

# Count tool calls by looking at messages with tool_calls
lg_tool_calls = [
    m for m in lg_result["messages"]
    if hasattr(m, 'tool_calls') and m.tool_calls
]
lg_tool_names = []
for m in lg_tool_calls:
    for tc in m.tool_calls:
        lg_tool_names.append(tc["name"])

print(f"\n🔧 Tools called: {lg_tool_names}")
print(f"⏱️  Latency: {lg_time:.1f}s")
print(f"💬 Messages in history: {len(lg_result['messages'])}")
print(f"\n📤 Answer:\n{lg_answer}")

### LangGraph Bonus: Checkpointing (State Persistence)

One of LangGraph's killer features is **Checkpointing** — the ability to save
and restore agent state automatically after every node execution.

**Why does this matter?**
- If the agent crashes mid-execution, you can resume from the last checkpoint
- You can implement **time travel** — go back to any previous state
- You can implement **Human-in-the-Loop** — pause, get approval, resume

In Lab 01's raw agent, state was lost if anything failed.
LangGraph gives you persistence for free with one parameter: `checkpointer`.

Below we use `MemorySaver` (in-memory checkpointer for demos).
In production, you'd use `PostgresSaver` or `SqliteSaver`.

In [ ]:
# ──────────────────────────────────────────────────────
# LangGraph with Checkpointing.
#
# By adding checkpointer="memory", every node execution
# is automatically saved. This enables:
# - Resume after crash
# - Multi-turn conversations (thread_id)
# - Time travel debugging
#
# The thread_id groups messages into a conversation.
# Same thread_id = agent remembers previous messages.
# Different thread_id = fresh conversation.
# ──────────────────────────────────────────────────────

from langgraph.checkpoint.memory import MemorySaver

# Create agent WITH checkpointing
lg_agent_persistent = create_react_agent(
    model=llm,
    tools=lg_tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),  # ← This one parameter adds persistence!
)

# Conversation 1: Tell the agent something
config = {"configurable": {"thread_id": "demo-thread-1"}}
r1 = lg_agent_persistent.invoke(
    {"messages": [{"role": "user", "content": "My name is Roy and I work in Tel Aviv."}]},
    config=config,
)
print("Turn 1:", r1["messages"][-1].content[:100])

# Conversation 2: Ask about something it should remember
r2 = lg_agent_persistent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather where I work?"}]},
    config=config,  # Same thread_id → agent remembers!
)
print("\nTurn 2:", r2["messages"][-1].content[:200])

print("\n✅ Checkpointing works! The agent remembered your city across turns.")
print("   In production, replace MemorySaver() with PostgresSaver() for durability.")

---

# 🏗️ STAGE 2: Microsoft Agent Framework (Semantic Kernel)

## What is it?

**Microsoft Agent Framework** is Microsoft's unified, production-grade SDK.
In Python, the package is `semantic-kernel`. It takes a fundamentally different
approach from LangGraph:

| Concept | LangGraph | Microsoft Agent Framework |
|---------|-----------|---------------------------|
| **Agent model** | State Graph (nodes + edges) | Kernel + Plugins |
| **Tools** | `@tool` decorated functions | `@kernel_function` in Plugin classes |
| **Orchestration** | Graph routing | Automatic function calling via Kernel |
| **State** | Shared state dictionary | Kernel manages state internally |
| **Languages** | Python, JS | Python, C#, Java |
| **Best for** | Complex workflows with custom routing | Enterprise, Azure-first teams |

### The Core Idea: Kernel + Plugins

```
┌─────────────────────────────────────────────────────┐
│                    🔮 Kernel                         │
│                                                      │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐          │
│  │ Weather  │  │  Math    │  │Knowledge │  Plugins  │
│  │ Plugin   │  │  Plugin  │  │ Plugin   │          │
│  └──────────┘  └──────────┘  └──────────┘          │
│                                                      │
│  ┌──────────────────────┐                           │
│  │  Azure OpenAI        │  AI Connector             │
│  │  (GPT-4.1)           │                           │
│  └──────────────────────┘                           │
│                                                      │
│  The Kernel sees all plugins, sends their schemas   │
│  to the LLM, and executes whichever function the    │
│  LLM chooses — automatically.                       │
└─────────────────────────────────────────────────────┘
```

**Key difference from LangGraph:** You don't define a graph or loop. The Kernel
handles tool dispatch automatically via `function_choice_behavior="auto"`.
You just define Plugins (groups of functions) and invoke a prompt.

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 2: Microsoft Agent Framework (Semantic Kernel)
#
# What this cell does:
# 1. Creates a Kernel — the central orchestrator
# 2. Adds an Azure OpenAI chat service to the Kernel
# 3. Defines Plugins using @kernel_function decorator
#    (Plugins are the SK equivalent of LangGraph's @tool)
# 4. Adds Plugins to the Kernel
#
# The Kernel will auto-discover all @kernel_function
# methods and expose them to the LLM as callable tools.
# ──────────────────────────────────────────────────────

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.functions import kernel_function

# Step 1: Create the Kernel and connect it to Azure OpenAI
sk_kernel = Kernel()
sk_kernel.add_service(
    AzureChatCompletion(
        deployment_name=MODEL,
        endpoint=AZURE_ENDPOINT,
        api_key=AZURE_API_KEY,
    )
)

# Step 2: Define Plugins.
# In SK, a Plugin is a Python class where each method
# decorated with @kernel_function becomes a tool.
# The 'description' parameter is what the LLM reads to
# decide when to call the function (same as Lab 01's schemas).

class WeatherPlugin:
    """Plugin for weather information."""
    @kernel_function(description="Get the current weather for a city")
    def get_weather(self, city: str) -> str:
        return get_weather(city)

class MathPlugin:
    """Plugin for mathematical calculations."""
    @kernel_function(description="Evaluate a mathematical expression")
    def calculate(self, expression: str) -> str:
        return calculate(expression)

class KnowledgePlugin:
    """Plugin for searching the company knowledge base."""
    @kernel_function(description="Search company knowledge base for policies and procedures")
    def search_knowledge(self, query: str) -> str:
        return search_knowledge(query)

# Step 3: Register Plugins with the Kernel.
# The string name ("weather", etc.) is how the LLM
# refers to the plugin in its function calls.
sk_kernel.add_plugin(WeatherPlugin(), "weather")
sk_kernel.add_plugin(MathPlugin(), "math")
sk_kernel.add_plugin(KnowledgePlugin(), "knowledge")

print("✅ Semantic Kernel agent created")
print(f"   Plugins: weather, math, knowledge")
print(f"   Functions: {[f.name for f in sk_kernel.get_list_of_function_metadata()]}")

### Run the SK Agent

Unlike LangGraph where you call `agent.invoke()`, in Semantic Kernel you call
`kernel.invoke_prompt()` with `function_choice_behavior="auto"`.

**What does `auto` mean?**
The Kernel sends ALL registered plugin functions to the LLM as available tools.
The LLM decides which to call. The Kernel automatically executes the chosen
function and sends the result back to the LLM. This repeats until the LLM
gives a text answer — the same ReAct loop, but managed by the Kernel.

In [ ]:
# ──────────────────────────────────────────────────────
# Run Semantic Kernel on the same test question.
#
# invoke_prompt() handles the full ReAct loop:
# 1. Sends prompt + all plugin schemas to LLM
# 2. LLM returns tool_call → Kernel executes it
# 3. Kernel sends result back to LLM
# 4. Repeat until LLM returns text
#
# function_choice_behavior="auto" tells the Kernel
# to let the LLM decide which functions to call.
# ──────────────────────────────────────────────────────
import asyncio
from semantic_kernel.connectors.ai.open_ai import AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

async def run_sk_agent(question: str) -> str:
    """Run the Semantic Kernel agent on a question."""
    settings = AzureChatPromptExecutionSettings(
        function_choice_behavior=FunctionChoiceBehavior.Auto()
    )
    result = await sk_kernel.invoke_prompt(
        prompt=f"{SYSTEM_PROMPT}\n\nUser: {question}",
        settings=settings,
    )
    return str(result)

print("📊 STAGE 2: Microsoft Agent Framework (Semantic Kernel)")
print("=" * 60)

start = time.time()
sk_answer = await run_sk_agent(TEST_QUESTION)
sk_time = time.time() - start

print(f"\n⏱️  Latency: {sk_time:.1f}s")
print(f"\n📤 Answer:\n{sk_answer}")

### SK Bonus: Multi-Agent with Agent Framework

Semantic Kernel's Agent Framework allows you to create **multiple specialized agents**
that collaborate. This is the evolution from AutoGen:

```
┌─────────────────────────────────────────────────────┐
│                 Agent Group Chat                     │
│                                                      │
│  🔍 Research Agent  →  📊 Analyst Agent              │
│  (has search tools)    (has math tools)              │
│         ↓                    ↓                      │
│     finds data          analyzes it                 │
│         ↓                    ↓                      │
│              ✍️ Writer Agent                        │
│              (synthesizes report)                   │
└─────────────────────────────────────────────────────┘
```

Below is a conceptual example. Multi-agent requires more setup,
but the key insight is: SK treats agents as first-class citizens
that can hand off work to each other.

In [ ]:
# ──────────────────────────────────────────────────────
# SK Multi-Agent Example (Conceptual)
#
# This demonstrates the API pattern for creating
# specialized agents that collaborate.
#
# In production, each agent would have different tools
# and instructions, and the AgentGroupChat would
# coordinate their conversation.
# ──────────────────────────────────────────────────────

from semantic_kernel.agents import ChatCompletionAgent

# Create specialized agents — each with its own role
researcher = ChatCompletionAgent(
    kernel=sk_kernel,
    name="Researcher",
    instructions="You are a research specialist. Find relevant data using the knowledge base.",
)

analyst = ChatCompletionAgent(
    kernel=sk_kernel,
    name="Analyst",  
    instructions="You are a data analyst. Use calculations to analyze data provided by the researcher.",
)

# In production, you'd create an AgentGroupChat:
# chat = AgentGroupChat(agents=[researcher, analyst])
# async for message in chat.invoke("Analyze our expense trends"):
#     print(message)

print("✅ Multi-agent pattern defined")
print(f"   Agents: {researcher.name}, {analyst.name}")
print("   In production: AgentGroupChat coordinates turn-taking")
print("   Key insight: Each agent can have DIFFERENT tools and instructions")

---

# 🏗️ STAGE 3: Deep Agents — The Autonomous Harness

## What is Deep Agents?

**Deep Agents** is LangChain's "agent harness" — built **on top of** LangGraph.
It's not a competitor; it's a **layer above** that adds capabilities
specifically designed for complex, long-running tasks.

### What Makes Deep Agents Different?

| Feature | LangGraph | Deep Agents |
|---------|-----------|-------------|
| **Planning** | You build it yourself | Built-in `write_todos` tool |
| **Context management** | Manual | Virtual filesystem (ls, read, write) |
| **Subagents** | You design the graph | Built-in `task` tool spawns subagents |
| **Long-term memory** | Add via Store | Integrated memory store |
| **Setup** | Build graph from scratch | `create_deep_agent()` — one function |

### The Mental Model

Think of it this way:
- **LangGraph** = LEGO bricks. You build exactly what you want, piece by piece.
- **Deep Agents** = A pre-built LEGO robot. It already has legs, arms, and a brain.
  You can customize it, but the basic structure is ready.

### How Deep Agents Manages Context

```
┌─────────────────────────────────────────────────────┐
│               Deep Agent Runtime                     │
│                                                      │
│  📋 Planning          Your question is complex?      │
│     write_todos()     Break it into steps first.     │
│                                                      │
│  📁 Filesystem        Tool returned 10,000 lines?    │
│     write_file()      Save to virtual file,          │
│     read_file()       read only what you need.       │
│                                                      │
│  🧑‍💻 Subagents        Need a specialist?             │
│     task()            Spawn a sub-agent with its     │
│                       own context window.            │
│                                                      │
│  💾 Memory            Remember across conversations? │
│     LangGraph Store   Persistent key-value memory.   │
└─────────────────────────────────────────────────────┘
```

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 3: Deep Agents
#
# What this cell does:
# 1. Uses create_deep_agent() — the one-liner agent factory
# 2. Passes our same shared tools as plain functions
#    (Deep Agents accepts raw functions — no decorator needed!)
# 3. The agent automatically gets EXTRA built-in tools:
#    - write_todos: for task planning and decomposition
#    - ls, read_file, write_file: virtual filesystem
#    - task: spawn subagents for complex subtasks
#
# Notice: we pass the model as "azure-openai:{deployment}"
# Deep Agents uses LangChain model strings internally.
# ──────────────────────────────────────────────────────

from deepagents import create_deep_agent

# Create the deep agent.
# Unlike LangGraph where we needed @tool wrappers,
# Deep Agents accepts plain Python functions directly!
da_agent = create_deep_agent(
    model=llm,  # Reuse the same AzureChatOpenAI instance from Stage 1
    tools=[get_weather, calculate, search_knowledge],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Deep Agent created")
print(f"   Model: {MODEL}")
print(f"   Custom tools: get_weather, calculate, search_knowledge")
print(f"   Built-in tools: write_todos, ls, read_file, write_file, edit_file, task")

### Run the Deep Agent

The Deep Agent's `invoke()` method works like LangGraph's but with
extra capabilities running behind the scenes.

**Watch for these differences:**
- The agent might use `write_todos` to plan the steps before executing tools
- The agent might use `write_file` to save intermediate results
- The agent has more autonomy — it decides not just *which* tool, but *how* to organize work

In [ ]:
# ──────────────────────────────────────────────────────
# Run Deep Agent on the same test question.
#
# What happens under the hood:
# 1. Agent receives the question
# 2. May use write_todos to decompose into steps
# 3. Calls our tools (weather, knowledge, calculate)
# 4. May write intermediate results to virtual filesystem
# 5. Synthesizes final answer
#
# The invoke() API is identical to LangGraph (since
# Deep Agents is built on top of LangGraph).
# ──────────────────────────────────────────────────────

print("📊 STAGE 3: Deep Agents")
print("=" * 60)

start = time.time()

da_result = da_agent.invoke(
    {"messages": [{"role": "user", "content": TEST_QUESTION}]}
)

da_time = time.time() - start

# Extract the final answer
da_answer = da_result["messages"][-1].content

# Count tool calls (including built-in tools)
da_tool_calls = [
    m for m in da_result["messages"]
    if hasattr(m, 'tool_calls') and m.tool_calls
]
da_tool_names = []
for m in da_tool_calls:
    for tc in m.tool_calls:
        da_tool_names.append(tc["name"])

print(f"\n🔧 Tools called: {da_tool_names}")
print(f"⏱️  Latency: {da_time:.1f}s")
print(f"💬 Messages in history: {len(da_result['messages'])}")
print(f"\n📤 Answer:\n{da_answer}")

### Deep Agent Bonus: Task Planning

Deep Agents' `write_todos` built-in tool lets the agent break complex tasks
into discrete steps and track progress. Let's give it a more complex question
to see planning in action.

**Why does planning matter?**
For simple 3-step questions (like our test), the LLM can handle it in its head.
But for 10+ step tasks ("analyze our entire expense policy, compare it to
industry benchmarks, and write a 5-page recommendation"), planning prevents
the agent from losing track of where it is.

In [ ]:
# ──────────────────────────────────────────────────────
# Deep Agent with a complex question to trigger planning.
#
# This question is deliberately multi-faceted to see if
# the Deep Agent uses write_todos to decompose it.
#
# Watch the tool calls — you may see:
# 1. write_todos (plan the steps)
# 2. search_knowledge (find policies)
# 3. get_weather (check multiple cities)
# 4. calculate (compare budgets)
# ──────────────────────────────────────────────────────

complex_question = (
    "I need to plan business trips to both Tokyo and London next month. "
    "For each city: what's the weather? What's the hotel budget? "
    "If Tokyo hotel is $180/night for 3 nights and London is $195/night for 2 nights, "
    "what's my total hotel cost? Am I within the per-night policy limit?"
)

print("📊 Deep Agent — Complex Task with Planning")
print("=" * 60)
print(f"Question: {complex_question}\n")

da_complex = da_agent.invoke(
    {"messages": [{"role": "user", "content": complex_question}]}
)

# Show which tools were used (including built-in ones)
for m in da_complex["messages"]:
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f"   🔧 {tc['name']}({json.dumps(tc.get('args', {}))[:80]})")

print(f"\n📤 Answer:\n{da_complex['messages'][-1].content}")

---

# 🏗️ STAGE 4: MCP — The Universal Tool Standard

## What is MCP?

**MCP (Model Context Protocol)** is an open standard created by Anthropic that
lets you build a tool **once** and connect it to **any** AI framework.

### The Problem MCP Solves

```
WITHOUT MCP:                          WITH MCP:
                                      
Build Slack tool for LangGraph         Build Slack MCP Server ONCE
Build Slack tool for SK                     │
Build Slack tool for Deep Agents       ┌────┼────┐────────┐
Build Slack tool for Claude            │    │    │        │
                                       LG   SK   DA    Claude
= 4x the work                         = 1x the work
```

### How MCP Works

MCP uses a client-server architecture:
- **MCP Server**: Exposes tools via a standard protocol (like a REST API for AI tools)
- **MCP Client**: Any framework that speaks MCP can discover and call those tools

Think of it like **USB**: before USB, every device had its own connector.
USB standardized the interface. MCP does the same for AI tools.

### What We'll Build

We'll create a simple MCP tool server that exposes our `get_weather` function,
and then show how any framework can connect to it.

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 4: MCP Tool Server
#
# What this cell does:
# 1. Defines an MCP server using the mcp Python SDK
# 2. Registers our weather function as an MCP tool
# 3. Shows how the tool schema is exposed via MCP protocol
#
# In production, this server would run as a separate
# process (e.g., a Docker container, Azure Function,
# or even a VS Code extension). Any MCP-compatible
# client can then discover and call this tool.
#
# Note: In a notebook we show the server definition.
# Running a full MCP server requires a separate process,
# so we demonstrate the schema and pattern here.
# ──────────────────────────────────────────────────────

# This is how you DEFINE an MCP tool server:
mcp_server_code = '''
from mcp.server.fastmcp import FastMCP

# Create an MCP server
mcp = FastMCP("Acme Corp Tools")

# Register a tool — any MCP client can now discover and call it
@mcp.tool()
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    weather_data = {
        "tel aviv": {"temp": 28, "condition": "sunny", "humidity": 65},
        "new york": {"temp": 12, "condition": "cloudy", "humidity": 70},
        "london":   {"temp": 8,  "condition": "rainy",  "humidity": 85},
        "tokyo":    {"temp": 20, "condition": "clear",  "humidity": 55},
    }
    data = weather_data.get(city.lower(), {"temp": 15, "condition": "unknown", "humidity": 50})
    return f"Weather in {city}: {data[\\"temp\\"]}°C, {data[\\"condition\\"]}"

@mcp.tool()
def search_knowledge(query: str) -> str:
    """Search company knowledge base for policies."""
    # ... same as our shared function
    return "Employees get 22 vacation days per year."

# Run the server (stdio transport for local, SSE for remote)
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

print("📋 MCP Server Definition:")
print("=" * 60)
print(mcp_server_code)
print("=" * 60)
print("\n✅ Key points about MCP:")
print("   1. @mcp.tool() auto-generates the JSON schema from the function signature")
print("   2. Any MCP client (Claude Desktop, VS Code, LangChain) can discover these tools")
print("   3. The server runs as a separate process — tools are isolated and secure")
print("   4. Transport options: stdio (local), SSE (remote/HTTP)")

### How Frameworks Connect to MCP

Each framework has its own way to connect to MCP servers.
The key insight: **you build the MCP server once, and every framework
can use it without any changes to the server code.**

| Framework | How It Connects to MCP |
|-----------|------------------------|
| **LangGraph** | `langchain-mcp-adapters` package converts MCP tools to LangChain tools |
| **Semantic Kernel** | Built-in `MCPServerPlugin` loads MCP tools as SK functions |
| **Deep Agents** | Inherits LangGraph's MCP support transparently |
| **Claude Desktop** | Native MCP support in settings |
| **VS Code Copilot** | MCP servers configurable in workspace settings |

In [ ]:
# ──────────────────────────────────────────────────────
# How to USE an MCP server from different frameworks.
#
# These are the connection patterns for each framework.
# Notice: the MCP server code stays EXACTLY the same.
# Only the client-side integration differs.
# ──────────────────────────────────────────────────────

print("🔌 MCP Client Integration Patterns")
print("=" * 60)

print("""
┌─────────────────────────────────────────────────────────┐
│ LangGraph + MCP:                                        │
│                                                         │
│   from langchain_mcp_adapters import MCPToolkit          │
│                                                         │
│   toolkit = MCPToolkit(server="./weather_server.py")    │
│   tools = toolkit.get_tools()  # Auto-discovers tools   │
│   agent = create_react_agent(model=llm, tools=tools)    │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│ Semantic Kernel + MCP:                                  │
│                                                         │
│   kernel.add_plugin(                                    │
│       MCPServerPlugin(server="./weather_server.py"),    │
│       "mcp_weather"                                     │
│   )                                                     │
│   # MCP tools appear as regular SK functions!           │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│ Claude Desktop + MCP (config file):                     │
│                                                         │
│   {                                                     │
│     "mcpServers": {                                     │
│       "weather": {                                      │
│         "command": "python",                            │
│         "args": ["weather_server.py"]                   │
│       }                                                 │
│     }                                                   │
│   }                                                     │
└─────────────────────────────────────────────────────────┘
""")

print("💡 Key takeaway: MCP = build once, connect everywhere.")
print("   This is why MCP is becoming the industry standard for AI tools.")

---

# 🏗️ STAGE 5: Framework Comparison

Now the moment of truth — let's compare all three frameworks side by side.

We've run the **same question** through **all three frameworks** using the
**same model** (GPT-4.1) and the **same tools**. Any differences we see
come from the framework itself — not the model or data.

In [ ]:
# ──────────────────────────────────────────────────────
# STAGE 5: Side-by-Side Comparison
#
# This cell builds a comparison table from the results
# we collected in Stages 1-3.
#
# We compare:
# - Latency (speed)
# - Tool calls (did it use all 3 tools?)
# - Code complexity (lines of setup code)
# - Answer quality (visual inspection)
# ──────────────────────────────────────────────────────

print("═" * 70)
print("📊 FRAMEWORK COMPARISON REPORT")
print("═" * 70)

print(f"\n📝 Test Question: {TEST_QUESTION[:80]}...")
print(f"🧠 Model: {MODEL} (same for all)")

# Comparison Table
print("\n" + "─" * 70)
print(f"{'Metric':<25} {'LangGraph':<15} {'SK/MAF':<15} {'Deep Agents':<15}")
print("─" * 70)
print(f"{'Latency':<25} {lg_time:<15.1f} {sk_time:<15.1f} {da_time:<15.1f}")
print(f"{'Tool calls':<25} {len(lg_tool_names):<15} {'(auto)':<15} {len(da_tool_names):<15}")
print(f"{'Tools used':<25} {','.join(lg_tool_names)[:13]:<15} {'(auto)':<15} {','.join(da_tool_names)[:13]:<15}")
print(f"{'Setup lines':<25} {'~15':<15} {'~25':<15} {'~10':<15}")
print(f"{'Checkpointing':<25} {'Built-in':<15} {'Manual':<15} {'Built-in':<15}")
print(f"{'Multi-agent':<25} {'Via subgraph':<15} {'Native':<15} {'Via task()':<15}")
print(f"{'Planning':<25} {'Manual':<15} {'Via Planner':<15} {'Built-in':<15}")
print(f"{'MCP Support':<25} {'Adapter':<15} {'Native':<15} {'Inherited':<15}")
print("─" * 70)

In [ ]:
# ──────────────────────────────────────────────────────
# Answer Quality Comparison
#
# Display the actual answers from all three frameworks
# so we can visually compare quality, completeness,
# and grounding.
# ──────────────────────────────────────────────────────

print("\n" + "═" * 70)
print("📤 ANSWER COMPARISON")
print("═" * 70)

print(f"\n🦜 LangGraph Answer ({lg_time:.1f}s):")
print("─" * 40)
print(lg_answer[:500])

print(f"\n🔮 SK/MAF Answer ({sk_time:.1f}s):")
print("─" * 40)
print(str(sk_answer)[:500])

print(f"\n🔬 Deep Agents Answer ({da_time:.1f}s):")
print("─" * 40)
print(da_answer[:500])

In [ ]:
# ──────────────────────────────────────────────────────
# Decision Guide: Which Framework Should You Choose?
#
# This cell prints a practical decision guide based
# on what we learned from running all three frameworks.
# ──────────────────────────────────────────────────────

print("\n" + "═" * 70)
print("🧭 DECISION GUIDE: Which Framework?")
print("═" * 70)

print("""
┌─────────────────────────────────────────────────────────────────────┐
│                                                                     │
│  Choose LANGGRAPH when:                                             │
│  ✅ You need full control over the agent's execution flow           │
│  ✅ You have complex workflows with branching and cycles            │
│  ✅ You want the largest open-source ecosystem (LangChain)          │
│  ✅ You need production checkpointing (Postgres, SQLite)            │
│  ✅ You're Python-first and want maximum flexibility                │
│                                                                     │
│  Choose MICROSOFT AGENT FRAMEWORK (SK) when:                        │
│  ✅ You're in the Microsoft / Azure ecosystem                       │
│  ✅ You need C# AND Python support (multi-language teams)           │
│  ✅ You want native multi-agent orchestration                       │
│  ✅ You're migrating from Semantic Kernel or AutoGen                │
│  ✅ Enterprise compliance and governance are critical               │
│                                                                     │
│  Choose DEEP AGENTS when:                                           │
│  ✅ You need autonomous agents for complex, multi-step tasks        │
│  ✅ Your agent works with large files, codebases, or documents      │
│  ✅ You want built-in planning (write_todos)                        │
│  ✅ You need subagent spawning for context isolation                 │
│  ✅ You're building a coding agent or terminal assistant             │
│                                                                     │
│  Use MCP EVERYWHERE when:                                           │
│  ✅ You want tools that work across ALL frameworks                   │
│  ✅ You're building a tool library for your organization             │
│  ✅ You want to avoid framework lock-in for your tools               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
""")

print("🎉 Lab 07 complete!")

---

# 🎓 Summary: What We Built and Learned

### The Same Agent, Three Ways

```
Same Question + Same Model + Same Tools
    ↓              ↓            ↓
LangGraph     SK / MAF     Deep Agents
(15 lines)    (25 lines)   (10 lines)
    ↓              ↓            ↓
Graph-based   Plugin-based  Harness-based
    ↓              ↓            ↓
Same quality answers — different DX
```

### Key Concepts

| Concept | What It Means |
|---------|---------------|
| **LangGraph** | Graph-based agent runtime — nodes, edges, state, checkpointing |
| **Semantic Kernel / MAF** | Microsoft's unified agent SDK — Kernel, Plugins, multi-agent |
| **Deep Agents** | Agent harness with built-in planning, filesystem, and subagents |
| **MCP** | Model Context Protocol — universal tool standard (build once, use everywhere) |
| **Checkpointing** | Saving agent state after each step for recovery and time travel |
| **Plugin** | SK's unit of tool organization (a class with @kernel_function methods) |
| **Agent Harness** | Pre-built runtime layer that wraps an LLM with tools and context management |

### Framework Evolution

| What We Explored | Production Pattern |
|------------------|-------------------|
| LangGraph `create_react_agent` | Custom StateGraph with 10+ nodes |
| SK `invoke_prompt` | `AgentGroupChat` with multiple specialized agents |
| Deep Agents `create_deep_agent` | Customized harness with sandboxed code execution |
| MCP server definition | MCP servers deployed as Azure Functions or containers |

### What's Next?

In **Lab 08**, we'll add **Observability** — tracing every LLM call, tool execution,
and token usage so you can debug and optimize your agents in production.

---

> **[← Back to Lab 06](../lab-06-evaluation/README.md)** | **[→ Lab 08: Observability](../lab-08-observability/README.md)**